# Large Assignment 02 - Image Dataset

## 1. Setup, EDA-Informed Preprocessing, and Feature Extraction (ResNet-18)

In [10]:
# 1. Setup, EDA-Informed Preprocessing, and Feature Extraction (ResNet-18)
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import numpy as np
import os
import json

# 1. Device Setup for M2 Max
# Using MPS (Metal Performance Shaders) for 30 GPU cores
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Preprocessing Informed by previous EDA
# Values sourced directly from EDA outputs [Section 4]
mean = [0.4416, 0.4461, 0.4718] 
std = [0.2040, 0.2081, 0.2058]

# Training transforms include augmentation for cluttered backgrounds
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
    # Note: HorizontalFlip is excluded per EDA findings on 6/9 and 2/5 ambiguity
])

# Test transforms only include normalization
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# 3. Load Datasets
train_set = datasets.SVHN(root='./data', split='train', download=True, transform=train_transform)
test_set = datasets.SVHN(root='./data', split='test', download=True, transform=test_transform)

# Recommended batch size for M2 Max with 32GB Ram
train_loader = DataLoader(train_set, batch_size=256, shuffle=False, num_workers=0)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=0)

def extract_features(dataloader, model):
    model.eval()
    all_features = []
    all_labels = []
    
    with torch.no_grad():
        for batch_idx, (inputs, labels) in enumerate(dataloader):
            inputs = inputs.to(device)
            # Forward pass through frozen backbone
            features = model(inputs)
            # Flatten features to 1D vector
            features = features.view(features.size(0), -1)
            
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())
            
            if (batch_idx + 1) % 50 == 0:
                print(f"Processed batch {batch_idx + 1}/{len(dataloader)}")
    return np.concatenate(all_features), np.concatenate(all_labels)

# Initialize ResNet-18 as a feature extractor
print("Initializing ResNet-18 for feature extraction...")
resnet18 = models.resnet18(weights='DEFAULT')
# Remove the final fully connected layer to get raw features
resnet18 = nn.Sequential(*list(resnet18.children())[:-1])
resnet18 = resnet18.to(device)

# Execute Extraction
print("Extracting training features...")
X_train_features, y_train = extract_features(train_loader, resnet18)
print("Extracting test features...")
X_test_features, y_test = extract_features(test_loader, resnet18)

# Save to disk as planned
os.makedirs('features', exist_ok=True)
np.save('features/resnet18_train_32.npy', X_train_features)
np.save('features/resnet18_test_32.npy', X_test_features)
np.save('features/train_labels.npy', y_train)
np.save('features/test_labels.npy', y_test)

print(f"Features saved! Shape: {X_train_features.shape}")

Using device: mps
Initializing ResNet-18 for feature extraction...
Extracting training features...
Processed batch 50/287
Processed batch 100/287
Processed batch 150/287
Processed batch 200/287
Processed batch 250/287
Extracting test features...
Processed batch 50/102
Processed batch 100/102
Features saved! Shape: (73257, 512)


## 2. Classical ML on Pretrained Features

In [11]:
# 2. Classical ML on Pretrained Features
import time
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# 1. Load the features saved in Block 1
X_train = np.load('features/resnet18_train_32.npy')
X_test = np.load('features/resnet18_test_32.npy')
y_train = np.load('features/train_labels.npy')
y_test = np.load('features/test_labels.npy')

# 2. Define the "Tournament" of models
# Note: use 'balance' weights to address the 3.03x imbalance found in the EDA
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Linear SVM": LinearSVC(class_weight='balanced', max_iter=2000),
    "Random Forest": RandomForestClassifier(n_estimators=100, n_jobs=-1),
    "MLP (Neural Baseline)": MLPClassifier(hidden_layer_sizes=(256,), max_iter=20),
    "Gaussian Naive Bayes": GaussianNB()
}

ml_results = []

print(f"{'Model':<25} | {'Accuracy':<10} | {'Macro F1':<10} | {'Time (s)':<10}")
print("-" * 65)

# 3. Train and Evaluate each model
for name, clf in classifiers.items():
    start_time = time.time()
    
    # Train the model
    clf.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    # Make predictions
    y_pred = clf.predict(X_test)
    
    # Calculate metrics (Macro F1 is critical due to class imbalance)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    cm = confusion_matrix(y_test, y_pred).tolist()
    
    print(f"{name:<25} | {acc:<10.4f} | {f1:<10.4f} | {train_time:<10.2f}")
    
    # Store for JSON export later
    ml_results.append({
        "model_name": name,
        "accuracy": float(acc),
        "macro_f1": float(f1),
        "train_time_s": float(train_time),
        "confusion_matrix": cm
    })

# 4. Save results for your webpage
with open('ml_section1_results_32.json', 'w') as f:
    json.dump(ml_results, f, indent=2)

Model                     | Accuracy   | Macro F1   | Time (s)  
-----------------------------------------------------------------


/Users/vnnguyen/Desktop/HK252/P4AIDS/EDA-Techniques-for-AI-and-DS/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression       | 0.4246     | 0.4196     | 38.37     
Linear SVM                | 0.4312     | 0.4216     | 189.00    
Random Forest             | 0.3547     | 0.3296     | 7.36      


/Users/vnnguyen/Desktop/HK252/P4AIDS/EDA-Techniques-for-AI-and-DS/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP (Neural Baseline)     | 0.4641     | 0.4490     | 5.92      
Gaussian Naive Bayes      | 0.2323     | 0.2327     | 0.09      


## 1.5 Feature Extraction: ResNet-18 with 224x224 Resize (Resolution Fix)

In [6]:
# 1.5 Setup, EDA-Informed Preprocessing, and Feature Extraction (ResNet-18) - Fixed Feature Extraction (224×224)
from torchvision import models as tv_models
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import numpy as np
import os
import json

# 1. Device Setup for M2 Max
# Using MPS (Metal Performance Shaders) for 30 GPU cores
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Preprocessing Informed by previous EDA
# Values sourced directly from EDA outputs [Section 4]
mean = [0.4416, 0.4461, 0.4718] 
std = [0.2040, 0.2081, 0.2058]

# Change 1: One unified transform for both train and test
extract_transform = transforms.Compose([
    transforms.Resize(224),          # ← THE KEY FIX: ResNet-18 expects 224×224
    transforms.ToTensor(),
    transforms.Normalize(mean, std)  # same EDA values as before
    # No augmentation — feature extraction needs deterministic, stable outputs
])

# 3. Load Datasets
# Change 2: Both train and test use the same extract_transform
train_set_feat = datasets.SVHN(root='./data', split='train', download=False, transform=extract_transform)
test_set_feat  = datasets.SVHN(root='./data', split='test',  download=False, transform=extract_transform)


train_loader_feat = DataLoader(train_set_feat, batch_size=128, shuffle=False, num_workers=0)
#                                              ↑ reduced from 256 because 224×224 images are 
#                                                49× larger in memory than 32×32

test_loader_feat = DataLoader(test_set_feat, batch_size=128, shuffle=False, num_workers=0)


def extract_features(dataloader, model):
    model.eval()
    all_features = []
    all_labels = []
    
    with torch.no_grad():
        for batch_idx, (inputs, labels) in enumerate(dataloader):
            inputs = inputs.to(device)
            # Forward pass through frozen backbone
            features = model(inputs)
            # Flatten features to 1D vector
            features = features.view(features.size(0), -1)
            
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())
            
            if (batch_idx + 1) % 50 == 0:
                print(f"Processed batch {batch_idx + 1}/{len(dataloader)}")
    return np.concatenate(all_features), np.concatenate(all_labels)

# Initialize ResNet-18 as a feature extractor
print("Initializing ResNet-18 for feature extraction...")
resnet18 = tv_models.resnet18(weights='DEFAULT')
# Remove the final fully connected layer to get raw features
resnet18 = nn.Sequential(*list(resnet18.children())[:-1])
resnet18 = resnet18.to(device)

# Execute Extraction
# Change 3: Save to different filenames so Block 1 files are preserved
print("Extracting training features...")
X_train_features, y_train = extract_features(train_loader_feat, resnet18)
print("Extracting test features...")
X_test_features, y_test   = extract_features(test_loader_feat,  resnet18)

# Save to disk as planned
os.makedirs('features', exist_ok=True)
np.save('features/resnet18_train_224.npy', X_train_features)  # ← different filename
np.save('features/resnet18_test_224.npy',  X_test_features)
np.save('features/train_labels.npy', y_train)   # labels don't change, can overwrite
np.save('features/test_labels.npy',  y_test)

print(f"Features saved! Shape: {X_train_features.shape}")
# Should still print (73257, 512) — same 512 features, but now properly computed

Using device: mps
Initializing ResNet-18 for feature extraction...
Extracting training features...
Processed batch 50/573
Processed batch 100/573
Processed batch 150/573
Processed batch 200/573
Processed batch 250/573
Processed batch 300/573
Processed batch 350/573
Processed batch 400/573
Processed batch 450/573
Processed batch 500/573
Processed batch 550/573
Extracting test features...
Processed batch 50/204
Processed batch 100/204
Processed batch 150/204
Processed batch 200/204
Features saved! Shape: (73257, 512)


## 2.5 Classical ML on 224x224 Features (Improve Results)

In [3]:
# 2.5 - Classical ML on Fixed Features (224×224)
import time
import os
import json
import joblib
import numpy as np
import torch
from torch import nn
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    confusion_matrix
) # Added precision and recall for the 8-metric table

# 1. Load the features saved in Block 1
# Change 1: Load the new 224×224 feature files
X_train = np.load('features/resnet18_train_224.npy')  # ← _224 files
X_test  = np.load('features/resnet18_test_224.npy')

y_train = np.load('features/train_labels.npy')
y_test = np.load('features/test_labels.npy')

# 2. Define the "Tournament" of models
# Note: use 'balance' weights to address the 3.03x imbalance found in the EDA
# Change 2: Fix MLP convergence
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Linear SVM":          LinearSVC(class_weight='balanced', max_iter=2000),
    "Random Forest":       RandomForestClassifier(n_estimators=100, n_jobs=-1),
    "MLP (Neural Baseline)": MLPClassifier(hidden_layer_sizes=(256,), max_iter=200), # ← 20→200
    "Gaussian Naive Bayes":  GaussianNB()
}


ml_results = []

# Header for the 8-Metric Tournament
print(f"{'Model':<25} | {'Acc (%)':<8} | {'Prec (%)':<8} | {'Rec (%)':<8} | {'F1-M':<8} | {'Trn(s)':<8} | {'Inf(ms)':<8} | {'Size'}")
print("-" * 110)

# 3. Train and Evaluate each model
for name, clf in classifiers.items():
    # 3.1. Measure Training Time
    start_train = time.time()
    clf.fit(X_train, y_train)
    train_time = time.time() - start_train
    
    # 3.2. Measure Inference Time (Latency)
    # We predict the entire test set and calculate time per image
    start_inf = time.time()
    y_pred = clf.predict(X_test)
    inf_time_total = time.time() - start_inf
    inf_latency_ms = (inf_time_total / len(y_test)) * 1000 # Convert to ms per image
    
    # 3.3. Calculate All Metrics (Macro average due to 3.03x imbalance)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro')
    rec = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro') 
    
    # 3.4. Measure Model Size
    # Save to a temporary file to see the actual footprint
    temp_path = f"temp_model.joblib"
    joblib.dump(clf, temp_path)
    model_size_kb = os.path.getsize(temp_path) / 1024
    os.remove(temp_path)
    print(f"{name:<25} | {acc*100:<8.2f} | {prec*100:<8.2f} | {rec*100:<8.2f} | {f1:<8.4f} | {train_time:<8.2f} | {inf_latency_ms:<8.2f} | {model_size_kb:.2f} KB")
    
    # 3.5. Store Extended Metrics for JSON (needed for image_ml.html)
    ml_results.append({
        "model_name": name,
        "accuracy": float(acc),
        "precision": float(prec),
        "recall": float(rec),
        "macro_f1": float(f1),
        "train_time_s": float(train_time),
        "inf_latency_ms": float(inf_latency_ms),
        "model_size_kb": float(model_size_kb),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist()
    })

# 4. Save results for the webpage
with open('ml_section1_results_224.json', 'w') as f:
    json.dump(ml_results, f, indent=2)

Model                     | Acc (%)  | Prec (%) | Rec (%)  | F1-M     | Trn(s)   | Inf(ms)  | Size
--------------------------------------------------------------------------------------------------------------


/Users/vnnguyen/Desktop/HK252/P4AIDS/EDA-Techniques-for-AI-and-DS/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression       | 62.48    | 60.64    | 62.47    | 0.6127   | 37.74    | 0.00     | 40.98 KB
Linear SVM                | 62.76    | 60.80    | 61.88    | 0.6121   | 126.17   | 0.00     | 40.86 KB
Random Forest             | 47.75    | 53.52    | 40.37    | 0.4219   | 13.32    | 0.00     | 530159.17 KB


/Users/vnnguyen/Desktop/HK252/P4AIDS/EDA-Techniques-for-AI-and-DS/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP (Neural Baseline)     | 60.80    | 59.10    | 59.49    | 0.5912   | 56.24    | 0.00     | 2103.90 KB
Gaussian Naive Bayes      | 39.72    | 40.56    | 36.61    | 0.3729   | 0.10     | 0.01     | 40.87 KB


## 3. Pipeline Comparison

In [13]:
# 3.1 Prepare Remaining Feature Sets (MobileNetV2 & Raw Pixels)
import os
import time
import joblib
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


print("--- PART 1: PREPARING ADDITIONAL FEATURES ---")

# 1. Initialize MobileNetV2 (Using the tv_models alias from Block 1.5)
print("Initializing MobileNetV2 for feature extraction...")
mobilenet = tv_models.mobilenet_v2(weights='DEFAULT')
# MobileNetV2 has a specific structure; replacing the classifier with Identity gives the 1280-d pooled features
mobilenet.classifier = nn.Identity()
mobilenet = mobilenet.to(device)

# 2. Extract MobileNetV2 Features (using the 224x224 loaders from Block 1.5)
print("Extracting MobileNetV2 features (takes a few minutes on M2 Max)...")
X_train_mbv2, _ = extract_features(train_loader_feat, mobilenet)
X_test_mbv2, _ = extract_features(test_loader_feat, mobilenet)

np.save('features/mobilenet_train_224.npy', X_train_mbv2)
np.save('features/mobilenet_test_224.npy', X_test_mbv2)
print(f"MobileNetV2 Features saved! Shape: {X_train_mbv2.shape}")

# 3. Prepare Raw Pixels (using the 32x32 loaders from Block 1)
print("\nPreparing Raw Pixel baselines (32x32)...")
def extract_raw_pixels(dataloader):
    all_pixels = []
    for inputs, _ in dataloader:
        # Flatten the 3x32x32 images into 3072-d vectors
        flat = inputs.view(inputs.size(0), -1).numpy()
        all_pixels.append(flat)
    return np.concatenate(all_pixels)

X_train_raw = extract_raw_pixels(train_loader)
X_test_raw = extract_raw_pixels(test_loader)
print(f"Raw Pixels ready! Shape: {X_train_raw.shape}")


print("\n--- PART 2: THE 8-PIPELINE TOURNAMENT (EXTENDED METRICS) ---")

# Load ResNet features (already saved from Block 1.5)
X_train_rn18 = np.load('features/resnet18_train_224.npy')
X_test_rn18 = np.load('features/resnet18_test_224.npy')
y_train = np.load('features/train_labels.npy')
y_test = np.load('features/test_labels.npy')

# Base classifiers (with balanced weights based on your EDA)
lr_base = LogisticRegression(max_iter=1000, class_weight='balanced')
svm_base = LinearSVC(class_weight='balanced', max_iter=2000)
rf_base = RandomForestClassifier(n_estimators=100, n_jobs=-1)

# Define the 8 Pipelines
pipelines = [
    {"id": 1, "name": "ResNet18 -> None -> LR", "X_tr": X_train_rn18, "X_te": X_test_rn18,
     "pipe": Pipeline([('clf', clone(lr_base))])},
    
    {"id": 2, "name": "ResNet18 -> PCA-128 -> LR", "X_tr": X_train_rn18, "X_te": X_test_rn18,
     "pipe": Pipeline([('pca', PCA(n_components=128)), ('clf', clone(lr_base))])},
    
    {"id": 3, "name": "ResNet18 -> PCA-128 -> SVM", "X_tr": X_train_rn18, "X_te": X_test_rn18,
     "pipe": Pipeline([('pca', PCA(n_components=128)), ('clf', clone(svm_base))])},
    
    {"id": 4, "name": "ResNet18 -> PCA-128 -> RF", "X_tr": X_train_rn18, "X_te": X_test_rn18,
     "pipe": Pipeline([('pca', PCA(n_components=128)), ('clf', clone(rf_base))])},
    
    {"id": 5, "name": "MobileNetV2 -> None -> LR", "X_tr": X_train_mbv2, "X_te": X_test_mbv2,
     "pipe": Pipeline([('clf', clone(lr_base))])},
    
    {"id": 6, "name": "MobileNetV2 -> PCA-256 -> LR", "X_tr": X_train_mbv2, "X_te": X_test_mbv2,
     "pipe": Pipeline([('pca', PCA(n_components=256)), ('clf', clone(lr_base))])},
    
    {"id": 7, "name": "Raw Pixels -> PCA-128 -> LR", "X_tr": X_train_raw, "X_te": X_test_raw,
     "pipe": Pipeline([('pca', PCA(n_components=128)), ('clf', clone(lr_base))])},
    
    {"id": 8, "name": "Raw Pixels -> PCA-128 -> SVM", "X_tr": X_train_raw, "X_te": X_test_raw,
     "pipe": Pipeline([('pca', PCA(n_components=128)), ('clf', clone(svm_base))])}
]

pipeline_results = []

# Updated Print Header
print(f"{'ID':<3} | {'Pipeline Name':<30} | {'Acc (%)':<8} | {'Prec (%)':<8} | {'Rec (%)':<8} | {'F1-M':<8} | {'Trn(s)':<8} | {'Inf(ms)':<8} | {'Size'}")
print("-" * 115)

for p in pipelines:
    start_train = time.time()
    p["pipe"].fit(p["X_tr"], y_train)
    train_time = time.time() - start_train
    
    # Measure Inference Latency (ms per image)
    start_inf = time.time()
    y_pred = p["pipe"].predict(p["X_te"])
    inf_latency_ms = ((time.time() - start_inf) / len(y_test)) * 1000
    
    # Calculate Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro')
    rec = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')
    cm = confusion_matrix(y_test, y_pred).tolist()
    
    # Measure Size
    temp_path = "temp_pipe.joblib"
    joblib.dump(p["pipe"], temp_path)
    size_kb = os.path.getsize(temp_path) / 1024
    os.remove(temp_path)
    
    print(f"{p['id']:<3} | {p['name']:<30} | {acc*100:<8.2f} | {prec*100:<8.2f} | {rec*100:<8.2f} | {f1:<8.4f} | {train_time:<8.2f} | {inf_latency_ms:<8.2f} | {size_kb:.1f} KB")
    
    pipeline_results.append({
        "rank": 0, # Sorted in JS
        "pipeline": p["name"],
        "accuracy": float(acc),
        "precision": float(prec),
        "recall": float(rec),
        "macro_f1": float(f1),
        "train_time_s": float(train_time),
        "inf_latency_ms": float(inf_latency_ms),
        "model_size_kb": float(size_kb),
        "confusion_matrix": cm
    })
    
# Save results to JSON for the webpage dashboard
with open('ml_section2_pipelines.json', 'w') as f:
    json.dump(pipeline_results, f, indent=2)

--- PART 1: PREPARING ADDITIONAL FEATURES ---
Initializing MobileNetV2 for feature extraction...
Extracting MobileNetV2 features (takes a few minutes on M2 Max)...
Processed batch 50/573
Processed batch 100/573
Processed batch 150/573
Processed batch 200/573
Processed batch 250/573
Processed batch 300/573
Processed batch 350/573
Processed batch 400/573
Processed batch 450/573
Processed batch 500/573
Processed batch 550/573
Processed batch 50/204
Processed batch 100/204
Processed batch 150/204
Processed batch 200/204
MobileNetV2 Features saved! Shape: (73257, 1280)

Preparing Raw Pixel baselines (32x32)...
Raw Pixels ready! Shape: (73257, 3072)

--- PART 2: THE 8-PIPELINE TOURNAMENT (EXTENDED METRICS) ---
ID  | Pipeline Name                  | Acc (%)  | Prec (%) | Rec (%)  | F1-M     | Trn(s)   | Inf(ms)  | Size
-------------------------------------------------------------------------------------------------------------------


/Users/vnnguyen/Desktop/HK252/P4AIDS/EDA-Techniques-for-AI-and-DS/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


1   | ResNet18 -> None -> LR         | 62.48    | 60.64    | 62.47    | 0.6127   | 38.59    | 0.00     | 41.1 KB
2   | ResNet18 -> PCA-128 -> LR      | 56.65    | 54.96    | 56.62    | 0.5545   | 1.28     | 0.00     | 271.3 KB
3   | ResNet18 -> PCA-128 -> SVM     | 57.01    | 54.90    | 55.78    | 0.5512   | 20.77    | 0.00     | 271.2 KB
4   | ResNet18 -> PCA-128 -> RF      | 45.01    | 52.07    | 37.19    | 0.3910   | 7.86     | 0.00     | 579455.8 KB


/Users/vnnguyen/Desktop/HK252/P4AIDS/EDA-Techniques-for-AI-and-DS/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


5   | MobileNetV2 -> None -> LR      | 57.92    | 55.54    | 57.07    | 0.5600   | 61.88    | 0.00     | 101.1 KB
6   | MobileNetV2 -> PCA-256 -> LR   | 53.64    | 51.52    | 52.87    | 0.5180   | 3.35     | 0.00     | 1309.8 KB
7   | Raw Pixels -> PCA-128 -> LR    | 17.73    | 16.95    | 17.07    | 0.1643   | 4.38     | 0.00     | 1561.3 KB
8   | Raw Pixels -> PCA-128 -> SVM   | 20.71    | 18.30    | 17.90    | 0.1740   | 37.96    | 0.00     | 1561.2 KB


## 4. Fine-Tuning ResNet-18

In [14]:
# 4. Transfer Learning (Fine-Tuning) Tournament
import time
import copy
import os
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.nn as nn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("--- BLOCK 4: DEEP TRANSFER LEARNING TOURNAMENT ---")

# 1. Calculate Class Weights (from EDA Imbalance)
# Using class counts from EDA: digit 1 dominates (19.1%), digit 9 is minority (6.3%)
class_counts = np.array([6692, 18960, 14734, 11379, 9981, 9266, 7704, 7614, 6705, 6254])
total_samples = class_counts.sum()
num_classes = len(class_counts)
weights = total_samples / (num_classes * class_counts)
class_weights = torch.FloatTensor(weights).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

# 2. Define 5 Model Configurations (ordered fastest-first)
# freeze strategy: "head_only" | "partial" | False (full fine-tune)
configurations = [
    {"id": "rn18_head",    "name": "ResNet-18 (Head-Only)",         "arch": "resnet18",     "freeze": "head_only"},
    {"id": "mbv2_head",    "name": "MobileNetV2 (Head-Only)",       "arch": "mobilenet_v2", "freeze": "head_only"},
    {"id": "rn18_partial", "name": "ResNet-18 (Partial Fine-Tune)", "arch": "resnet18",     "freeze": "partial"},
    {"id": "mbv2_full",    "name": "MobileNetV2 (Full Fine-Tune)",  "arch": "mobilenet_v2", "freeze": False},
    {"id": "rn18_full",    "name": "ResNet-18 (Full Fine-Tune)",    "arch": "resnet18",     "freeze": False},
]

# Epoch count per strategy — head-only converges fast, full fine-tune needs more
epoch_map = {
    "head_only": 5,   # ~5 min per model
    "partial":   8,   # ~20 min
    False:       10   # ~55 min per model
}

# 3. Model Factory — creates and freezes layers based on strategy
def create_model(arch, freeze_strategy):
    if arch == "resnet18":
        model = tv_models.resnet18(weights='DEFAULT')

        if freeze_strategy == "head_only":
            # Freeze entire backbone — only the new fc layer will train
            for param in model.parameters():
                param.requires_grad = False

        elif freeze_strategy == "partial":
            # Freeze entire backbone first
            for param in model.parameters():
                param.requires_grad = False
            # Then unfreeze layer4 (last residual block) — learns task-specific features
            for param in model.layer4.parameters():
                param.requires_grad = True
            # fc will be replaced below and is trainable by default

        # False = full fine-tune, nothing frozen — all layers adapt to SVHN

        # Replace ImageNet 1000-class head with SVHN 10-class head
        # nn.Linear always creates trainable parameters regardless of freeze strategy
        num_ftrs = model.fc.in_features  # 512 for ResNet-18
        model.fc = nn.Linear(num_ftrs, 10)

    elif arch == "mobilenet_v2":
        model = tv_models.mobilenet_v2(weights='DEFAULT')

        if freeze_strategy == "head_only":
            for param in model.parameters():
                param.requires_grad = False

        elif freeze_strategy == "partial":
            # Freeze entire backbone first
            for param in model.parameters():
                param.requires_grad = False
            # Unfreeze last two conv blocks (features[17] and features[18])
            # These are the highest-level feature extractors before classification
            for param in model.features[17].parameters():
                param.requires_grad = True
            for param in model.features[18].parameters():
                param.requires_grad = True

        # Replace classifier head: 1280-d → 10 classes
        num_ftrs = model.classifier[1].in_features  # 1280 for MobileNetV2
        model.classifier[1] = nn.Linear(num_ftrs, 10)

    return model.to(device)

# 4. Universal Training Loop
def train_model(model, criterion, optimizer, scheduler, num_epochs):
    start_time = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'  Epoch {epoch+1}/{num_epochs}', end=' ')

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                dataloader = train_loader_feat  # 224×224 loaders from Block 1.5
            else:
                model.eval()
                dataloader = test_loader_feat

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                # zero_grad() clears gradients from the previous batch.
                # Without this, gradients would accumulate across batches — wrong.

                with torch.set_grad_enabled(phase == 'train'):
                    # set_grad_enabled(True)  during train: track gradients for backprop
                    # set_grad_enabled(False) during val:   save memory, no backprop needed
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    # torch.max returns (values, indices) along dim=1
                    # _ discards the confidence scores, preds keeps the class index (0-9)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()   # compute gradients via backpropagation
                        optimizer.step()  # update weights using computed gradients

                running_loss += loss.item() * inputs.size(0)
                # loss.item() = average loss for this batch
                # multiply by batch size to get total loss for the batch
                # accumulate across all batches to compute epoch average later
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()
                # Update learning rate after each training phase.
                # CosineAnnealingLR smoothly reduces lr from 1e-4 toward 0 over T_max epochs.

            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_corrects.float() / len(dataloader.dataset)
            # .float() instead of .double() — MPS (Apple GPU) doesn't support float64

            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(float(epoch_acc))
                print(f'[Train: {epoch_acc:.3f}]', end=' ')
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(float(epoch_acc))
                print(f'[Val: {epoch_acc:.3f}]')

            # Save best model weights based on validation accuracy
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
                # deepcopy is critical — without it, best_model_wts would just be
                # a reference to model.state_dict() which keeps changing each epoch

    train_time = time.time() - start_time
    model.load_state_dict(best_model_wts)  # restore best checkpoint before returning
    return model, history, train_time

# 5. Execute the Tournament
finetune_results = []

print(f"\n{'Model':<35} | {'Acc':<7} | {'F1-M':<7} | {'Time(m)':<8} | {'Inf(ms)':<8} | {'Size(MB)'}")
print("-" * 85)

for config in configurations:
    print(f"\n{'='*50}")
    print(f"Training: {config['name']}...")
    print(f"{'='*50}")

    # Create fresh model for each configuration
    model = create_model(config['arch'], config['freeze'])

    # Count trainable parameters — useful for your report
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.1f}%)")

    # Only pass parameters that require gradients to optimizer
    # This is essential for head-only and partial strategies —
    # passing frozen params would waste memory and compute
    num_epochs = epoch_map[config['freeze']]
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-4
    )
    scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    # Train
    best_model, history, train_time = train_model(
        model, criterion, optimizer, scheduler, num_epochs
    )

    # Evaluate — measure inference latency on full test set
    best_model.eval()
    all_preds, all_labels = [], []

    start_inf = time.time()
    with torch.no_grad():
        for inputs, labels in test_loader_feat:
            inputs = inputs.to(device)
            outputs = best_model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    inf_latency_ms = ((time.time() - start_inf) / len(all_labels)) * 1000
    # Average milliseconds per image during inference

    # Calculate all metrics
    acc   = accuracy_score(all_labels, all_preds)
    prec  = precision_score(all_labels, all_preds, average='macro')
    rec   = recall_score(all_labels, all_preds, average='macro')
    f1    = f1_score(all_labels, all_preds, average='macro')
    cm    = confusion_matrix(all_labels, all_preds).tolist()
    per_class_f1 = f1_score(all_labels, all_preds, average=None).tolist()

    # Measure model file size on disk
    temp_path = "temp_dl_model.pth"
    torch.save(best_model.state_dict(), temp_path)
    size_mb = os.path.getsize(temp_path) / (1024 * 1024)
    os.remove(temp_path)

    print(f"\n{config['name']:<35} | {acc*100:<7.2f} | {f1:<7.4f} | {train_time/60:<8.1f} | {inf_latency_ms:<8.3f} | {size_mb:.1f}")

    finetune_results.append({
        "model_name":     config['name'],
        "model_id":       config['id'],
        "freeze_strategy": str(config['freeze']),
        "epochs":         num_epochs,
        "trainable_params": trainable_params,
        "total_params":   total_params,
        "accuracy":       float(acc),
        "precision":      float(prec),
        "recall":         float(rec),
        "macro_f1":       float(f1),
        "train_time_s":   float(train_time),
        "inf_latency_ms": float(inf_latency_ms),
        "model_size_mb":  float(size_mb),
        "per_class_f1":   per_class_f1,
        "confusion_matrix": cm,
        "history":        history
    })

# 6. Save all 5 results to one JSON file for the webpage
with open('ml_section3_finetune.json', 'w') as f:
    json.dump(finetune_results, f, indent=2)

print("\n" + "="*50)
print("Tournament complete!")
print("Results saved to ml_section3_finetune.json")
print("="*50)

--- BLOCK 4: DEEP TRANSFER LEARNING TOURNAMENT ---

Model                               | Acc     | F1-M    | Time(m)  | Inf(ms)  | Size(MB)
-------------------------------------------------------------------------------------

Training: ResNet-18 (Head-Only)...
Trainable params: 5,130 / 11,181,642 (0.0%)
  Epoch 1/5 [Train: 0.199] [Val: 0.251]
  Epoch 2/5 [Train: 0.311] [Val: 0.320]
  Epoch 3/5 [Train: 0.349] [Val: 0.350]
  Epoch 4/5 [Train: 0.364] [Val: 0.362]
  Epoch 5/5 [Train: 0.371] [Val: 0.367]

ResNet-18 (Head-Only)               | 36.69   | 0.3493  | 11.1     | 1.246    | 42.7

Training: MobileNetV2 (Head-Only)...
Trainable params: 12,810 / 2,236,682 (0.6%)
  Epoch 1/5 [Train: 0.244] [Val: 0.352]
  Epoch 2/5 [Train: 0.350] [Val: 0.394]
  Epoch 3/5 [Train: 0.378] [Val: 0.409]
  Epoch 4/5 [Train: 0.388] [Val: 0.416]
  Epoch 5/5 [Train: 0.392] [Val: 0.417]

MobileNetV2 (Head-Only)             | 41.72   | 0.3859  | 10.8     | 1.130    | 8.8

Training: ResNet-18 (Partial Fine-Tune)